# (1)

In [198]:
import pandas as pd
##WIP: add seaborn and matplotlib here.

#Do not want to tamper on the base data. 
#We'll save the changes in a new .csv file as one of the final steps.
birdstrikes_raw = pd.read_csv('../birdstrikes.csv', parse_dates=['Flight Date'])
birdstrikes = birdstrikes_raw.copy()

#Table size. (Should be 10000, 14)
print(f'Size/shape: {birdstrikes.shape}')

#Columns.
print(f'Columns: {list(birdstrikes.columns.values)}')

#General idea of the actual values we'll be dealing with.
print(birdstrikes.value_counts().head(10))

#These first 10 entries have 0 on three columns, being: Cost Other, Cost Repair, and Cost Total.
#Therefore, we'll conduct a quick check if this applies for all other entries as well.
print(len(birdstrikes[birdstrikes['Cost Other'] == 0])) #Count: 9928
print(len(birdstrikes[birdstrikes['Cost Repair'] == 0])) #Count: 9822
print(len(birdstrikes[birdstrikes['Cost Total $'] == 0])) #Count: 9791

Size/shape: (10000, 14)
Columns: ['Airport Name', 'Aircraft Make Model', 'Effect Amount of damage', 'Flight Date', 'Aircraft Airline Operator', 'Origin State', 'Phase of flight', 'Wildlife Size', 'Wildlife Species', 'Time of day', 'Cost Other', 'Cost Repair', 'Cost Total $', 'Speed IAS in knots']
Airport Name                    Aircraft Make Model  Effect Amount of damage  Flight Date  Aircraft Airline Operator  Origin State  Phase of flight  Wildlife Size  Wildlife Species       Time of day  Cost Other  Cost Repair  Cost Total $  Speed IAS in knots
NEW ORLEANS INTL                B-737-300            Substantial              1990-01-11   SOUTHWEST AIRLINES         Louisiana     Take-off run     Small          Rock pigeon            Day          0           0            0             140.0                 1
WASHINGTON DULLES INTL ARPT     BA-146               Medium                   1990-03-03   AIR WISCONSIN AIRLINES     DC            Climb            Large          Unknown bird - la

Seen in its entirety, a row describes the circumstances of the flight through a handful of different day:
What type of flight from where, and when the incident happened both in terms of day/night and flight phase.

Most values do not have any specified costs, between about 0.5-2.0% of the entries have anything beyond 0.
The zeroes do not have to mean a cost was never inserted, but actually that no cost to fix the plane in any way was needed.
Since there are birds, it is highly unlikely that any of them would weigh more than a human, and
collision between a human and a plane would highly likely and realistically only cause harm to the human in most likely.

We'll keep these values; they could be outliers of further interest.

# (2)

In [199]:
#To know if something could be incorrect, it can be smart to first get what datatypes there are.
birdstrikes.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Airport Name               10000 non-null  str           
 1   Aircraft Make Model        10000 non-null  str           
 2   Effect Amount of damage    1061 non-null   str           
 3   Flight Date                10000 non-null  datetime64[us]
 4   Aircraft Airline Operator  10000 non-null  str           
 5   Origin State               10000 non-null  str           
 6   Phase of flight            10000 non-null  str           
 7   Wildlife Size              10000 non-null  str           
 8   Wildlife Species           10000 non-null  str           
 9   Time of day                10000 non-null  str           
 10  Cost Other                 10000 non-null  int64         
 11  Cost Repair                10000 non-null  int64         
 12  Cost Total $    

**Speed IAS in knots** has 7164 non-null entries. 
**Effect amount of damage** has 1061 non-null entries; worth investigating.

Another point of curiosity is why only one of the Cost columns have a $ in it: **Cost Total \$**. 

In [200]:
#How many entries that are duplicates in the eyes of each other, 
#and how many duplicates there are (ignoring the first entry found).
print(birdstrikes.duplicated(keep=False).sum())
print(birdstrikes.duplicated().sum())

birdstrikes[birdstrikes.duplicated()]

45
24


,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots
341,ATLANTIC CITY INTL,F-16,NaN,1990-10-15,MILITARY,New Jersey,Landing Roll,Medium,Unknown bird or bat,Night,0,0,0,0.0
1114,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-14,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,150.0
1132,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-23,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,140.0
1268,GREATER PITTSBURGH,DC-9-30,NaN,1992-06-15,US AIRWAYS*,Pennsylvania,Climb,Small,Unknown bird - small,Day,0,0,0,150.0
1814,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1993-04-28,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,180.0
2081,CHARLESTON AFB/INTL ARPT,C-141B,NaN,1993-08-18,MILITARY,South Carolina,Climb,Medium,Unknown bird or bat,Day,0,0,0,130.0
2897,CHARLESTON AFB/INTL ARPT,C-141B,NaN,1994-10-10,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Night,0,0,0,160.0
3043,HONOLULU INTL ARPT,C-141,NaN,1995-01-10,MILITARY,Hawaii,Approach,Medium,Unknown bird - medium,Day,0,0,0,128.0
6190,CHARLESTON AFB/INTL ARPT,C-17A,NaN,1998-11-04,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Night,0,0,0,230.0
6236,CHARLESTON AFB/INTL ARPT,C-17A,NaN,1998-12-04,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Day,0,0,0,130.0


The duplicates that are identical on all columns will not give any new insights to us.
It is way too unlikely that entries matching on everything, from model, airport AND collision time, would happen.
We'll start off by deleting these.

In [201]:
#keep is 'first' by default, so we do not specify anything.
birdstrikes = birdstrikes.drop_duplicates(keep='first')

#Count as a safety measure.
print(birdstrikes.duplicated().sum())

0


Most birdstrikes have reoccurring aircraft models, based on the fact that this gives 9751 Trues.

In [202]:
birdstrikes.duplicated(subset = 'Aircraft Make Model').value_counts()

True     9751
False     225
Name: count, dtype: int64

A sample used as a check - that does not however answer about the entire dataset - shows that entries with the same models do not 
have too identical values over all the columns.

In [203]:
birdstrikes[birdstrikes.duplicated(subset = 'Aircraft Make Model')].head(20)

,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots
4,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-01-12,MILITARY,Louisiana,Climb,Medium,Unknown bird or bat,Day,0,0,0,160.0
5,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-02-02,MILITARY,Louisiana,Take-off run,Medium,Unknown bird or bat,Day,0,0,0,180.0
6,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-02-06,MILITARY,Louisiana,Climb,Medium,Unknown bird or bat,Dusk,0,0,0,140.0
8,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-03-01,MILITARY,Louisiana,Climb,Medium,Unknown bird or bat,Day,0,0,0,180.0
9,BARKSDALE AIR FORCE BASE ARPT,B-52,NaN,1990-03-02,MILITARY,Louisiana,Take-off run,Medium,Unknown bird or bat,Dusk,0,0,0,130.0
11,BARKSDALE AIR FORCE BASE ARPT,B-52,NaN,1990-03-08,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,150.0
13,CHARLESTON AFB/INTL ARPT,C-141B,NaN,1990-03-09,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Night,0,0,0,190.0
14,BARKSDALE AIR FORCE BASE ARPT,B-52,NaN,1990-03-09,MILITARY,Louisiana,Take-off run,Medium,Unknown bird or bat,Night,0,0,0,150.0
16,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-03-19,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Dusk,0,0,0,150.0
18,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-04-04,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,135.0


We can see that these few outliers in terms of classification, which we by ourselves cannot decode, pull the mean up a lot.
Looking at how much they add upp together, and in isolation, the mean gets skewed.
Although they would be of interesting consideration, we cannot say for sure what they are, and they 
make the others we do understand less valuable in comparison. 

Since they become noise for our case, we'll remove them (FOR NOW).
NOTE: This might be an incorrect way to treat this data, and so this may be looked over again later on.

In [204]:
birdstrikes['Effect Amount of damage'].value_counts(dropna=False)

Effect Amount of damage
NaN            8916
Minor           548
Substantial     311
Medium          186
C                14
B                 1
Name: count, dtype: int64

In [205]:
birdstrikes[
    birdstrikes['Effect Amount of damage'].isin(['B', 'C'])
]

,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots
299,ATLANTIC CITY INTL,C-137C,C,1990-09-26,MILITARY,New Jersey,Climb,Medium,Unknown bird - medium,Day,0,23656,23656,150.0
441,BARKSDALE AIR FORCE BASE ARPT,B-52G,C,1990-11-30,MILITARY,Louisiana,Climb,Small,Cattle egret,Day,0,67435,67435,175.0
598,BARKSDALE AIR FORCE BASE ARPT,B-52G,C,1991-05-14,MILITARY,Louisiana,Approach,Large,Turkey vulture,Day,0,160764,160764,140.0
648,CHARLESTON AFB/INTL ARPT,C-141B,C,1991-06-22,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Day,0,37234,37234,200.0
1493,GREATER PITTSBURGH,KC-135E,C,1992-09-12,MILITARY,Pennsylvania,Approach,Medium,Unknown bird - medium,Day,0,117399,117399,200.0
2543,ATLANTIC CITY INTL,F-16,C,1994-05-19,MILITARY,New Jersey,Approach,Medium,Unknown bird - medium,Day,0,20439,20439,180.0
3381,CHARLESTON AFB/INTL ARPT,C-17A,C,1995-08-04,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Day,0,69282,69282,230.0
5270,BARKSDALE AIR FORCE BASE ARPT,B-52H,C,1997-11-07,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Dusk,0,21514,21514,155.0
5753,BARKSDALE AIR FORCE BASE ARPT,B-52H,C,1998-07-30,MILITARY,Louisiana,Approach,Large,Great blue heron,Night,0,21360,21360,150.0
7407,CHARLESTON AFB/INTL ARPT,C-17A,C,2000-04-24,MILITARY,South Carolina,Climb,Medium,Unknown bird or bat,Day,0,32073,32073,NaN


In [206]:
#All: Mean is ~2051.7
birdstrikes[birdstrikes['Aircraft Airline Operator'] == 'MILITARY']['Cost Total $'].agg(['mean', 'median'])

mean      2051.683047
median       0.000000
Name: Cost Total $, dtype: float64

In [207]:
#Without the unclear classifications: Mean drops to ~186
print(
    birdstrikes[
        (birdstrikes['Aircraft Airline Operator'] == 'MILITARY') &
        (~birdstrikes['Effect Amount of damage'].isin(['B', 'C']))
    ]['Cost Total $'].agg(['mean', 'median'])
)

#B: Mean is ~1271.4
print(
    birdstrikes[
        (birdstrikes['Aircraft Airline Operator'] == 'MILITARY') &
        (~birdstrikes['Effect Amount of damage'].isin(['B']))
    ]['Cost Total $'].agg(['mean', 'median'])
)

#C: Mean is ~981.3
print(
    birdstrikes[
        (birdstrikes['Aircraft Airline Operator'] == 'MILITARY') &
        (~birdstrikes['Effect Amount of damage'].isin(['C']))
    ]['Cost Total $'].agg(['mean', 'median'])
)

mean      186.006258
median      0.000000
Name: Cost Total $, dtype: float64
mean      1271.420664
median       0.000000
Name: Cost Total $, dtype: float64
mean      981.28
median      0.00
Name: Cost Total $, dtype: float64


Removal of B and C.

In [208]:
birdstrikes = birdstrikes.drop(birdstrikes[birdstrikes['Effect Amount of damage'].isin(['B', 'C'])].index)

Check that they are removed. (Only ones existing now are 'Substantial', 'Medium', 'Minor')

In [209]:
birdstrikes['Effect Amount of damage'].unique()

<StringArray>
[nan, 'Substantial', 'Medium', 'Minor']
Length: 4, dtype: str

To check if there could be any correlation between the Cost columns, before we change the name of the Cost Final $ column,
we'll check different values with an aggregation.

In [210]:
birdstrikes.groupby('Effect Amount of damage')[['Cost Other', 'Cost Repair', 'Cost Total $']].agg(['mean', 'median'])

Cost Other           Cost Repair         \
                                 mean median           mean median   
Effect Amount of damage                                              
Medium                     780.634409    0.0    4555.000000    0.0   
Minor                     1106.395985    0.0    3812.728102    0.0   
Substantial              10823.922830    0.0  101912.070740    0.0   

                          Cost Total $         
                                  mean median  
Effect Amount of damage                        
Medium                     5335.634409    0.0  
Minor                      4919.124088    0.0  
Substantial              112735.993569    0.0

0 results where Repair and Other do not add up to the Total.

In [211]:
birdstrikes[birdstrikes['Cost Total $'] != birdstrikes['Cost Other'] + birdstrikes['Cost Repair']]

,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots


The means in Other and Repair are the sum of Total. Safe to say that they do relate to each other, and we can rename the column.

In [212]:
birdstrikes = birdstrikes.rename(columns={'Cost Total $' : 'Cost Total'})

Slight renaming of Effect Amount of damage to a few columns, while we're at it.

In [213]:
columns = {
    'Effect Amount of damage' : 'Effect Amount of Damage',
    'Phase of flight' : 'Phase of Flight',
    'Time of day' : 'Time of Day',
    'Speed IAS in knots' : 'Speed IAS in Knots'
}

birdstrikes = birdstrikes.rename(columns=columns)

In [214]:
birdstrikes.columns

Index(['Airport Name', 'Aircraft Make Model', 'Effect Amount of Damage',
       'Flight Date', 'Aircraft Airline Operator', 'Origin State',
       'Phase of Flight', 'Wildlife Size', 'Wildlife Species', 'Time of Day',
       'Cost Other', 'Cost Repair', 'Cost Total', 'Speed IAS in Knots'],
      dtype='str')

Lastly, we'll do cleanup on the names in columns **Airport Name** and **Aircraft Airline Operator**.
We already know here that a few exceptions, such as acronyms appear, and we'll deal with those after doing a general replacement treatment.

**Airport** in the name is redundant. International airports are usually called "International Airport" in actuality, 
which one does not specify. Will simply standardize these to be readable and without redundant contents.

In [215]:
birdstrikes['Airport Name'] = (
    birdstrikes['Airport Name']
    .str.title()
    .str.replace(
        {'Arpt' : '', 
        "Intl" : "International",
        'Natl' : 'National',
        'Laguardia Ny' : 'LaGuardia',
        '(Or)' : ''}
    )
    .str.strip()
)
birdstrikes['Airport Name'].unique()

<StringArray>
[                  'Barksdale Air Force Base',
                  'New Orleans International',
            'Washington Dulles International',
          'Ronald Reagan Washington National',
               'Charleston Afb/International',
             'Lambert-St Louis International',
                                  'LaGuardia',
            'Charlotte/Douglas International',
                  'Los Angeles International',
               'Chicago O'Hare International',
                     'Portland International',
    'Fort Lauderdale/Hollywood International',
                              'Houston-Hobby',
                  'San Antonio International',
                   'Sacramento International',
                          'Will Rogers World',
                      'Atlanta International',
                 'Indianapolis International',
            'Dallas/Fort Worth International',
                  'Kansas City International',
              'Mineta San Jose International',

In [218]:
birdstrikes['Aircraft Airline Operator'] = (
    birdstrikes['Aircraft Airline Operator']
    .str.title()
    .str.replace(
        {
            'Alleghenyairlines' : 'Allegheny Airlines', 'Expressjet (Continental Exprs)' : 'ExpressJet',
            '*' : '',
            'Us' : 'US', 'Fedex' : 'FedEx', 'Skywest' : 'SkyWest', 'Airtran' : 'AirTran',
            'Ups' : 'UPS', 'Abx' : 'ABX', 'Psa' : 'PSA'
        }
    )
)

birdstrikes['Aircraft Airline Operator'].unique()

<StringArray>
[               'Military',      'Southwest Airlines',
           'FedEx Express',  'Air Wisconsin Airlines',
   'Trans States Airlines',              'US Airways',
       'American Airlines',                 'Unknown',
         'United Airlines',         'Delta Air Lines',
                'Business',          'Aloha Airlines',
    'Trans World Airlines',              'ExpressJet',
            'UPS Airlines',            'PSA Airlines',
   'America West Airlines',    'Continental Airlines',
              'Government',      'Northwest Airlines',
         'Alaska Airlines',            'Hawaiian Air',
        'SkyWest Airlines',         'Comair Airlines',
             'Horizon Air',         'Mesaba Airlines',
              'Air Canada',         'Astar Air Cargo',
                'Pinnacle',      'Atlantic Southeast',
                 'ABX Air',      'Allegheny Airlines',
         'Privately Owned', 'American Eagle Airlines',
       'Piedmont Airlines',     'Chautauqua Airline

With this, we have cleaned everything up before we are looking at missing values.

# (3)

While for example Aircraft Airline Operator has Unknown in some entries, 
the ones we can see immediately having null entries are:

**Effect Amount of Damage** - 8916 entries
**Speed IAS in Knots** - 2829 entries

The values are of quite a substantial portion of all entries, so they will not be dropped, at least.
To be consistent with naming that has already been here, we'll fill the Effect Amount of Damage NaN entries with 'Unknown'.

In [ ]:
#Count what is missing in every column. 
#The whole point here is 'keep', 'drop' or 'fill'.
birdstrikes.isnull().sum()

Airport Name                    0
Aircraft Make Model             0
Effect Amount of Damage      8916
Flight Date                     0
Aircraft Airline Operator       0
Origin State                    0
Phase of Flight                 0
Wildlife Size                   0
Wildlife Species                0
Time of Day                     0
Cost Other                      0
Cost Repair                     0
Cost Total                      0
Speed IAS in Knots           2829
dtype: int64

In [231]:
print(birdstrikes['Effect Amount of Damage'].value_counts(dropna=False))

birdstrikes['Effect Amount of Damage'] = (
    birdstrikes['Effect Amount of Damage'].fillna('Unknown')
)

print(birdstrikes['Effect Amount of Damage'].value_counts(dropna=False))

Effect Amount of Damage
NaN            8916
Minor           548
Substantial     311
Medium          186
Name: count, dtype: int64
Effect Amount of Damage
Unknown        8916
Minor           548
Substantial     311
Medium          186
Name: count, dtype: int64


This alone does not say very much. The definitely most normal values are in an interval between 120-160 knots.
So, we'll check them in relation to flight phase, (because the model names are basically cryptic enough for us to not get any value from them as directly).

In [229]:
birdstrikes['Speed IAS in Knots'].value_counts()

Speed IAS in Knots
140.0    971
130.0    625
150.0    528
120.0    470
160.0    422
        ... 
62.0       1
7.0        1
246.0      1
107.0      1
98.0       1
Name: count, Length: 122, dtype: int64

Since we do not want to delete any values, the two options we have now are "keep" or "fill". 
The fill would mean filling with the median, which would misrepresent Descent, Parked and Taxi. 
We'll keep them as NaN, so they do not skew the distribution more than necessary for us to make any interesting observations.

In [234]:
birdstrikes.groupby('Phase of Flight')['Speed IAS in Knots'].agg(['mean', 'median'])

,mean,median
Phase of Flight,,
Approach,152.921515,140.0
Climb,178.194754,167.0
Descent,232.032051,250.0
Landing Roll,111.422914,115.0
Parked,0.000000,0.0
Take-off run,122.786325,125.0
Taxi,41.750000,15.0


# (4)

In [235]:
us_regions_raw = pd.read_csv('../us_regions.csv')
us_regions = us_regions_raw.copy() 

*WIP*: Say if something is missing or not. 

Should probably be left, because we want everything from our main, and only something if there is things on the right.

In [ ]:
#Check shape of main table as it is, and check it after as well.
print(birdstrikes.shape)

test_merge = birdstrikes.merge(us_regions, on='Origin State', how='left')

print(test_merge.shape)

# (9)

– When, in the flight and during the day, do the strikes happen?

– What does a strike cost, and what does a cost of zero mean?

– Does the number of reports over the years mean more strikes, or more reporting?
